In [1]:
import numpy as np
import pandas as pd

import xgboost as xgb
from math import exp
from scipy.stats import norm

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score

pd.set_option('display.max_columns',50)

In [2]:
detailed_regular = pd.read_csv('./data/MRegularSeasonDetailedResults.csv')
print("Detailed regular data shape -->", detailed_regular.shape)

Detailed regular data shape --> (118449, 34)


In [3]:
def select_features(df):
    # generating and selecting important features
    df['WEffFGPerc'] = 100 * (df['WFGM'] + (0.5 * df['WFGM3']))/df['WFGA']
    df['LEffFGPerc'] = 100 * (df['LFGM'] + (0.5 * df['LFGM3']))/df['LFGA']
    df['WFTPerc'] = 100 * df['WFTM']/(df['WFTA']+1)
    df['LFTPerc'] = 100 * df['LFTM']/(df['LFTA']+1)
    df['WTOPerc'] = 100 * df['WTO'] / (df['WFGA'] - df['WOR'] + df['WTO'] + (0.44 * df['WFTA']))
    df['LTOPerc'] = 100 * df['LTO'] / (df['LFGA'] - df['LOR'] + df['LTO'] + (0.44 * df['LFTA']))
    df['WORPerc'] = 100 * df['WOR'] / (df['WOR'] + df['LDR'])
    df['LORPerc'] = 100 * df['LOR'] / (df['LOR'] + df['WDR'])
    df['WDefEff'] = (df['WStl'] + df['WBlk'])/ df['WPF']
    df['LDefEff'] = (df['LStl'] + df['LBlk'])/ df['LPF']

    drop_cols = ['WLoc','NumOT','WFGM','WFGA','LFGM','LFGA','WFGM3','WFGA3','WStl','WBlk','WPF','LFGM3','LFGA3','WFTM','WFTA','LStl','LBlk','LPF','LFTM','LFTA','WOR','WDR','LOR','LDR','WTO','LTO']
    df.drop(drop_cols, inplace=True, axis=1)

    return df

# replace Wins by Team A and B logic
def assign_teams(df):
    # true means winner goes to Team A
    np.random.seed(123)
    coin = np.random.rand(len(df)) < 0.5

    # team A stats based on coin toss
    teamA = pd.DataFrame({
        'Season': df['Season'],
        'DayNum': df['DayNum'],
        'TeamID' : np.where(coin, df['WTeamID'], df['LTeamID']),
        'Score' :  np.where(coin, df['WScore'], df['LScore']),
        'ORPerc': np.where(coin, df['WORPerc'], df['LORPerc']),
        'Ast': np.where(coin, df['WAst'], df['LAst']),
        'TOPerc': np.where(coin, df['WTOPerc'], df['LTOPerc']),
        'DefEff': np.where(coin, df['WDefEff'], df['LDefEff']),
        'EffFG': np.where(coin, df['WEffFGPerc'], df['LEffFGPerc']),
        'FTPerc': np.where(coin, df['WFTPerc'], df['LFTPerc']),
    })

    # team B stats based on coin toss
    teamB = pd.DataFrame({
        'TeamID' : np.where(coin, df['LTeamID'], df['WTeamID']),
        'Score' :  np.where(coin, df['LScore'], df['WScore']),
        'ORPerc': np.where(coin, df['LORPerc'], df['WORPerc']),
        'Ast': np.where(coin, df['LAst'], df['WAst']),
        'TOPerc': np.where(coin, df['LTOPerc'], df['WTOPerc']),
        'DefEff': np.where(coin, df['LDefEff'], df['WDefEff']),
        'EffFG': np.where(coin, df['LEffFGPerc'], df['WEffFGPerc']),
        'FTPerc': np.where(coin, df['LFTPerc'], df['WFTPerc']),
    })

    # Create Win column: 1 if Team A gets the winner's stats, 0 otherwise
    win = pd.Series(np.where(coin, 1, 0), name='Win')

    # Optionally, rename columns to clearly indicate Team A and B stats (except for Season)
    teamA = teamA.rename(columns=lambda x: x if x in ['Season','DayNum'] else x + '_A')
    teamB = teamB.rename(columns=lambda x: x if x in ['Season','DayNum'] else x + '_B')

    # Concatenate the two teams' stats and the Win column into one DataFrame
    new_df = pd.concat([teamA, teamB, win], axis=1)
    
    return new_df

feature_df = select_features(detailed_regular)
final_df = assign_teams(feature_df)
final_df.head(10)

,Season,DayNum,TeamID_A,Score_A,ORPerc_A,Ast_A,TOPerc_A,DefEff_A,EffFG_A,FTPerc_A,TeamID_B,Score_B,ORPerc_B,Ast_B,TOPerc_B,DefEff_B,EffFG_B,FTPerc_B,Win
0,2003,10,1328,62,29.411765,8,25.466893,0.550000,43.396226,69.565217,1104,68,38.888889,13,30.699413,0.363636,49.137931,57.894737,0
1,2003,10,1272,70,37.500000,16,19.016969,0.444444,48.387097,50.000000,1393,63,41.666667,7,17.699115,0.875000,40.298507,42.857143,1
2,2003,11,1266,73,43.589744,15,15.683814,0.280000,48.275862,56.666667,1437,61,54.385965,9,18.714910,0.304348,32.191781,58.333333,1
3,2003,11,1457,50,47.222222,9,32.986111,0.304348,42.857143,50.000000,1296,56,23.076923,11,20.818876,0.888889,51.315789,53.125000,0
4,2003,11,1208,71,48.837209,12,15.903308,0.571429,43.548387,60.714286,1400,77,53.125000,12,21.971124,0.400000,54.098361,78.571429,0
5,2003,11,1458,81,35.294118,12,13.661202,0.666667,50.877193,82.142857,1186,55,20.000000,8,28.580024,0.280000,46.739130,66.666667,1
6,2003,12,1236,62,33.333333,11,40.365985,0.500000,51.219512,68.965517,1161,80,38.235294,14,22.321429,0.480000,43.636364,80.000000,0
7,2003,12,1457,61,18.604651,10,19.705728,1.222222,37.288136,70.833333,1186,75,34.210526,19,24.598654,0.428571,48.387097,68.181818,0
8,2003,12,1194,71,25.714286,9,22.997835,0.478261,52.586207,52.631579,1156,66,37.142857,13,32.946758,0.555556,51.923077,42.857143,1
9,2003,12,1458,84,37.837838,11,8.907363,0.923077,51.492537,75.000000,1296,56,29.032258,10,27.157514,0.222222,47.115385,53.846154,1


In [4]:
def compute_rolling_stats(df):
    # -------------------------------------------
    # STEP 1: Compute Differential Columns for Each Matchup
    # -------------------------------------------
    # For Team A perspective:
    df['Score_diff_A']   = df['Score_A']   - df['Score_B']
    df['ORPerc_diff_A']  = df['ORPerc_A']  - df['ORPerc_B']
    df['Ast_diff_A']     = df['Ast_A']     - df['Ast_B']
    df['TOPerc_diff_A']  = df['TOPerc_A']  - df['TOPerc_B']
    df['DefEff_diff_A']  = df['DefEff_A']  - df['DefEff_B']
    df['EffFG_diff_A']   = df['EffFG_A']   - df['EffFG_B']
    df['FTPerc_diff_A']  = df['FTPerc_A']  - df['FTPerc_B']
    
    # For Team B perspective (reverse the differential):
    df['Score_diff_B']   = df['Score_B']   - df['Score_A']
    df['ORPerc_diff_B']  = df['ORPerc_B']  - df['ORPerc_A']
    df['Ast_diff_B']     = df['Ast_B']     - df['Ast_A']
    df['TOPerc_diff_B']  = df['TOPerc_B']  - df['TOPerc_A']
    df['DefEff_diff_B']  = df['DefEff_B']  - df['DefEff_A']
    df['EffFG_diff_B']   = df['EffFG_B']   - df['EffFG_A']
    df['FTPerc_diff_B']  = df['FTPerc_B']  - df['FTPerc_A']
    
    # Create the regression target: Point Margin = Score_A - Score_B.
    df['Point_Margin'] = df['Score_A'] - df['Score_B']
    
    # -------------------------------------------
    # STEP 2: Convert Wide Data to Long Format
    # -------------------------------------------
    # For rolling computations it is easier if each row represents a single team's performance.
    
    # Create a DataFrame for Team A rows:
    df_A = df[['Season', 'DayNum', 'TeamID_A', 
               'Score_diff_A', 'ORPerc_diff_A', 'Ast_diff_A', 'TOPerc_diff_A',
               'DefEff_diff_A', 'EffFG_diff_A', 'FTPerc_diff_A']].copy()
    df_A.rename(columns={
        'TeamID_A': 'Team',
        'Score_diff_A': 'Score_diff',
        'ORPerc_diff_A': 'ORPerc_diff',
        'Ast_diff_A': 'Ast_diff',
        'TOPerc_diff_A': 'TOPerc_diff',
        'DefEff_diff_A': 'DefEff_diff',
        'EffFG_diff_A': 'EffFG_diff',
        'FTPerc_diff_A': 'FTPerc_diff'
    }, inplace=True)
    
    # Create a DataFrame for Team B rows:
    df_B = df[['Season', 'DayNum', 'TeamID_B', 
               'Score_diff_B', 'ORPerc_diff_B', 'Ast_diff_B', 'TOPerc_diff_B',
               'DefEff_diff_B', 'EffFG_diff_B', 'FTPerc_diff_B']].copy()
    df_B.rename(columns={
        'TeamID_B': 'Team',
        'Score_diff_B': 'Score_diff',
        'ORPerc_diff_B': 'ORPerc_diff',
        'Ast_diff_B': 'Ast_diff',
        'TOPerc_diff_B': 'TOPerc_diff',
        'DefEff_diff_B': 'DefEff_diff',
        'EffFG_diff_B': 'EffFG_diff',
        'FTPerc_diff_B': 'FTPerc_diff'
    }, inplace=True)
    
    # Combine Team A and Team B rows into one long DataFrame:
    df_long = pd.concat([df_A, df_B], ignore_index=True)
    
    # -------------------------------------------
    # STEP 3: Compute 7-Game Rolling Averages for Differential Stats
    # -------------------------------------------
    # Sort by Season, Team, and DayNum so that each team's games are in chronological order.
    df_long.sort_values(by=['Season', 'Team', 'DayNum'], inplace=True)
    
    # Define a function to compute rolling averages using a 7-game window.
    def compute_rolling(group, window=7):
        group = group.copy()
        group['Score_diff_7']  = group['Score_diff'].rolling(window=window, min_periods=window).mean()
        group['ORPerc_diff_7'] = group['ORPerc_diff'].rolling(window=window, min_periods=window).mean()
        group['Ast_diff_7']    = group['Ast_diff'].rolling(window=window, min_periods=window).mean()
        group['TOPerc_diff_7'] = group['TOPerc_diff'].rolling(window=window, min_periods=window).mean()
        group['DefEff_diff_7'] = group['DefEff_diff'].rolling(window=window, min_periods=window).mean()
        group['EffFG_diff_7']  = group['EffFG_diff'].rolling(window=window, min_periods=window).mean()
        group['FTPerc_diff_7'] = group['FTPerc_diff'].rolling(window=window, min_periods=window).mean()
        return group
    
    df_long = df_long.groupby(['Season', 'Team']).apply(compute_rolling, window=7)
    df_long.reset_index(drop=True, inplace=True)
    
    # -------------------------------------------
    # STEP 4: Merge Rolling Stats Back into the Original (Wide) Matchup-Level DataFrame
    # -------------------------------------------
    # We need to merge the rolling stats for Team A and Team B back into the original DataFrame.
    
    # Specify the columns from the long DataFrame that contain the rolling averages:
    rolling_cols = ['Season', 'DayNum', 'Team', 
                    'Score_diff_7', 'ORPerc_diff_7', 'Ast_diff_7', 'TOPerc_diff_7',
                    'DefEff_diff_7', 'EffFG_diff_7', 'FTPerc_diff_7']
    
    # Extract rolling stats for Team A:
    rolling_A = df_long[rolling_cols].copy()
    df = df.merge(rolling_A, left_on=['Season', 'DayNum', 'TeamID_A'],
                  right_on=['Season', 'DayNum', 'Team'], how='left', suffixes=('', '_A'))
    df.rename(columns={
        'Score_diff_7': 'Score_diff_7_A',
        'ORPerc_diff_7': 'ORPerc_diff_7_A',
        'Ast_diff_7': 'Ast_diff_7_A',
        'TOPerc_diff_7': 'TOPerc_diff_7_A',
        'DefEff_diff_7': 'DefEff_diff_7_A',
        'EffFG_diff_7': 'EffFG_diff_7_A',
        'FTPerc_diff_7': 'FTPerc_diff_7_A'
    }, inplace=True)
    df.drop('Team', axis=1, inplace=True)
    
    # Extract rolling stats for Team B:
    rolling_B = df_long[rolling_cols].copy()
    df = df.merge(rolling_B, left_on=['Season', 'DayNum', 'TeamID_B'],
                  right_on=['Season', 'DayNum', 'Team'], how='left', suffixes=('', '_B'))
    df.rename(columns={
        'Score_diff_7': 'Score_diff_7_B',
        'ORPerc_diff_7': 'ORPerc_diff_7_B',
        'Ast_diff_7': 'Ast_diff_7_B',
        'TOPerc_diff_7': 'TOPerc_diff_7_B',
        'DefEff_diff_7': 'DefEff_diff_7_B',
        'EffFG_diff_7': 'EffFG_diff_7_B',
        'FTPerc_diff_7': 'FTPerc_diff_7_B'
    }, inplace=True)
    df.drop('Team', axis=1, inplace=True)
    
    # -------------------------------------------
    # STEP 5: Create Final DataFrame and Drop Original Columns
    # -------------------------------------------
    # We keep only the keys, the rolling stats, and target variables.
    # removed Score_diff, Assists after checking correlation
    final_cols = ['Season', 'DayNum', 'TeamID_A', 'TeamID_B',
                  'ORPerc_diff_7_A', 'TOPerc_diff_7_A',
                  'DefEff_diff_7_A', 'EffFG_diff_7_A', 'FTPerc_diff_7_A',
                  'ORPerc_diff_7_B', 'TOPerc_diff_7_B',
                  'DefEff_diff_7_B', 'EffFG_diff_7_B', 'FTPerc_diff_7_B',
                  'Win', 'Point_Margin']
    
    df_final = df[final_cols].dropna().reset_index(drop=True)

    return (df_long[rolling_cols].copy(), df_final)

df_long, final_df = compute_rolling_stats(final_df)

C:\Users\siddh\AppData\Local\Temp\ipykernel_44216\2229484105.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_long = df_long.groupby(['Season', 'Team']).apply(compute_rolling, window=7)


In [5]:
df_long.tail()

,Season,DayNum,Team,Score_diff_7,ORPerc_diff_7,Ast_diff_7,TOPerc_diff_7,DefEff_diff_7,EffFG_diff_7,FTPerc_diff_7
236893,2025,103,1480,-6.714286,1.397058,-2.857143,-1.031445,0.174720,-7.891073,9.041144
236894,2025,106,1480,-12.142857,-0.773671,-4.000000,1.197422,-0.026187,-9.717385,2.350058
236895,2025,108,1480,-13.000000,-2.097104,-2.428571,2.082404,-0.066984,-8.437088,2.041668
236896,2025,112,1480,-12.428571,-2.501287,-1.142857,3.050552,-0.143247,-6.105439,-3.220364
236897,2025,114,1480,-10.714286,-0.426419,-1.142857,2.441679,-0.017003,-5.561034,-6.629455


In [6]:
final_df.head()

,Season,DayNum,TeamID_A,TeamID_B,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B,Win,Point_Margin
0,2003,33,1232,1183,-0.951985,-2.758769,0.120962,-4.983341,-12.428746,-2.439401,-2.613924,-0.132108,-7.448693,7.649894,1,6
1,2003,33,1237,1292,0.430207,2.448383,-0.130675,-15.615107,0.690647,0.975224,5.027691,-0.311933,-0.551846,-6.643701,0,-4
2,2003,35,1231,1435,0.979515,-0.298213,0.387308,8.557617,11.411400,0.485802,-1.763459,-0.014092,13.656050,-6.894016,1,17
3,2003,35,1250,1162,1.180624,3.828888,0.077071,1.456485,11.409258,-6.304264,1.474739,-0.283262,-11.607017,-10.361916,1,16
4,2003,37,1140,1360,0.619789,1.986108,0.035920,13.024963,10.488572,10.422564,6.703842,-0.103381,0.635117,-5.081380,1,15


In [7]:
train_data = final_df[final_df['Season'].isin(range(2003, 2025))].reset_index(drop=True)
train_data.to_csv('./data/final_train_data.csv', index=False)

In [8]:
# Define your features and target column
feature_cols = [
    'ORPerc_diff_7_A', 'TOPerc_diff_7_A', 'DefEff_diff_7_A',
    'EffFG_diff_7_A', 'FTPerc_diff_7_A', 'ORPerc_diff_7_B',
    'TOPerc_diff_7_B', 'DefEff_diff_7_B', 'EffFG_diff_7_B',
    'FTPerc_diff_7_B'
]
target_col = 'Point_Margin'

# Assuming train_data is your DataFrame
X = train_data[feature_cols]
y = train_data[target_col].values

# Define the conversion function from margin to probability
def margin_to_probability(margin_array, scale=10.0):
    """
    Convert predicted margins to a win probability for Team A
    using a Normal CDF approach:
        P(A wins) = Phi((margin) / scale)
    """
    return norm.cdf(margin_array / scale)

'''def margin_to_probability(margin_values):
    return 1.0 / (1.0 + np.exp(-margin_values))
'''

# Define the Brier score function
def brier_score(y_true, y_prob):
    return np.mean((y_true - y_prob)**2)

# Set up K-fold cross validation
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=123)

brier_scores = []
accuracy_scores = []

# Initialize the XGBoost regressor with your fixed parameters
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.6,
    colsample_bytree=0.6,
    reg_lambda=2,
    reg_alpha=1,
    random_state=123
)

for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
    # Create training and validation splits for the fold
    X_train_cv, X_val_cv = X.iloc[train_index], X.iloc[val_index]
    y_train_cv, y_val_cv = y[train_index], y[val_index]
    
    # Train the model
    model.fit(X_train_cv, y_train_cv)
    
    # Predict the continuous point margin on the validation fold
    y_val_pred_margin = model.predict(X_val_cv)
    
    # Convert predicted margins to win probabilities
    y_val_pred_prob = margin_to_probability(y_val_pred_margin)
    
    # Convert true margins to binary outcomes (win=1 if margin > 0, else 0)
    y_val_true_label = (y_val_cv > 0).astype(int)
    
    # For accuracy, threshold the predicted probabilities at 0.5
    y_val_pred_label = (y_val_pred_prob > 0.5).astype(int)
    
    # Compute Brier score for the fold
    fold_brier = brier_score(y_val_true_label, y_val_pred_prob)
    brier_scores.append(fold_brier)
    
    # Compute accuracy score for the fold
    fold_accuracy = accuracy_score(y_val_true_label, y_val_pred_label)
    accuracy_scores.append(fold_accuracy)
    
    print(f"Fold {fold} - Brier Score: {fold_brier:.4f}, Accuracy: {fold_accuracy:.4f}")

# Compute the average Brier score across folds
avg_brier = np.mean(brier_scores)
avg_accuracy = np.mean(accuracy_scores)
print(f"\nAverage Brier Score over {n_splits} folds: {avg_brier:.4f}")
print(f"Average Accuracy over {n_splits} folds: {avg_accuracy:.4f}")

Fold 1 - Brier Score: 0.1601, Accuracy: 0.7606
Fold 2 - Brier Score: 0.1640, Accuracy: 0.7558
Fold 3 - Brier Score: 0.1606, Accuracy: 0.7599
Fold 4 - Brier Score: 0.1660, Accuracy: 0.7497
Fold 5 - Brier Score: 0.1661, Accuracy: 0.7514
Fold 6 - Brier Score: 0.1606, Accuracy: 0.7629
Fold 7 - Brier Score: 0.1630, Accuracy: 0.7546
Fold 8 - Brier Score: 0.1622, Accuracy: 0.7606
Fold 9 - Brier Score: 0.1653, Accuracy: 0.7525
Fold 10 - Brier Score: 0.1612, Accuracy: 0.7587

Average Brier Score over 10 folds: 0.1629
Average Accuracy over 10 folds: 0.7567


In [9]:
team_data25 = df_long[df_long['Season']==2025].reset_index(drop=True)
team_data25

,Season,DayNum,Team,Score_diff_7,ORPerc_diff_7,Ast_diff_7,TOPerc_diff_7,DefEff_diff_7,EffFG_diff_7,FTPerc_diff_7
0,2025,5,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,12,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,16,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,21,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,22,1101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
10411,2025,103,1480,-6.714286,1.397058,-2.857143,-1.031445,0.174720,-7.891073,9.041144
10412,2025,106,1480,-12.142857,-0.773671,-4.000000,1.197422,-0.026187,-9.717385,2.350058
10413,2025,108,1480,-13.000000,-2.097104,-2.428571,2.082404,-0.066984,-8.437088,2.041668
10414,2025,112,1480,-12.428571,-2.501287,-1.142857,3.050552,-0.143247,-6.105439,-3.220364


In [10]:
def get_preds(df, TeamID_A, TeamID_B):
    feature_cols = ['ORPerc_diff_7','TOPerc_diff_7','DefEff_diff_7','EffFG_diff_7','FTPerc_diff_7']
    data_team_A = df[df['Team']==TeamID_A][feature_cols].tail(1).reset_index(drop=True).add_suffix('_A')
    data_team_B = df[df['Team']==TeamID_B][feature_cols].tail(1).reset_index(drop=True).add_suffix('_B')

    pred_data = pd.concat([data_team_A, data_team_B], axis=1)
    display(pred_data)

    preds_margin = model.predict(pred_data)
    preds_prob = margin_to_probability(preds_margin)
    return preds_prob

In [11]:
get_preds(df_long, 1106, 1384)

,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B
0,0.11645,-8.532967,0.325844,1.287475,15.133576,4.752603,2.66706,0.009612,3.71462,-11.450556


array([0.75090225])

In [12]:
teams = pd.read_csv('./data/MTeams.csv')
teams

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
...,...,...,...,...
375,1476,Stonehill,2023,2025
376,1477,East Texas A&M,2023,2025
377,1478,Le Moyne,2024,2025
378,1479,Mercyhurst,2025,2025


In [13]:
teams[teams['TeamName']=='Alabama St']

,TeamID,TeamName,FirstD1Season,LastD1Season
5,1106,Alabama St,1985,2025


In [14]:
teams[teams['TeamName'].str.contains('Francis')]

,TeamID,TeamName,FirstD1Season,LastD1Season
261,1362,San Francisco,1986,2025
282,1383,St Francis NY,1985,2023
283,1384,St Francis PA,1985,2025


In [15]:
teams[teams['TeamName'].str.contains('San Diego')]

,TeamID,TeamName,FirstD1Season,LastD1Season
259,1360,San Diego,1985,2025
260,1361,San Diego St,1985,2025
370,1471,UC San Diego,2021,2025


In [16]:
teams[teams['TeamName'].str.contains('North Carolina')]

,TeamID,TeamName,FirstD1Season,LastD1Season
213,1314,North Carolina,1985,2025


In [17]:
get_preds(df_long, 1361, 1314)

,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B
0,-3.781367,-3.27909,0.236272,4.910119,3.11676,8.323831,1.207848,-0.09806,9.331225,-5.13152


array([0.44479585])